## Modelo lineal

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

df = pd.read_csv('wfh_burnout_dataset.csv')

features = [
    'work_hours', 'screen_time_hours', 'meetings_count', 'breaks_taken',
    'after_hours_work', 'app_switches', 'sleep_hours', 'task_completion',
    'isolation_index', 'fatigue_score'
]

X = df[features]
y = df['burnout_risk']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Variables seleccionadas y etiquetas codificadas.")
print(f"Variables de entrada: {list(X.columns)}")

Variables seleccionadas y etiquetas codificadas.
Variables de entrada: ['work_hours', 'screen_time_hours', 'meetings_count', 'breaks_taken', 'after_hours_work', 'app_switches', 'sleep_hours', 'task_completion', 'isolation_index', 'fatigue_score']


In [ ]:
# División: 70% entrenamiento, 30% restante
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_encoded, test_size=0.30, random_state=42, stratify=y_encoded
)

# División del 30% restante: mitad para Validación (15%) y mitad para Test (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# Escalado de datos
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc = scaler.transform(X_val)
X_test_sc = scaler.transform(X_test)

print(f"Distribución de los datos:")
print(f"Entrenamiento (Train): {X_train.shape[0]} muestras")
print(f"Validación (Val):      {X_val.shape[0]} muestras")
print(f"Prueba (Test):         {X_test.shape[0]} muestras")

Distribución de los datos:
Entrenamiento (Train): 1400 muestras
Validación (Val):      300 muestras
Prueba (Test):         300 muestras


In [ ]:
# Crear y entrenar el modelo Regresión Logística
modelo_lineal = LogisticRegression(multi_class='multinomial', max_iter=1000)
modelo_lineal.fit(X_train_sc, y_train)

def evaluar_conjunto(X_sc, y_true, nombre):
    y_pred = modelo_lineal.predict(X_sc)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')
    print(f"RESULTADOS {nombre} ")
    print(f"Accuracy: {acc:.4f} ({acc*100:.2f}%)")
    print(f"F1-Score (Macro): {f1:.4f}\n")
    return acc, f1

acc_tr, f1_tr = evaluar_conjunto(X_train_sc, y_train, "TRAIN")
acc_va, f1_va = evaluar_conjunto(X_val_sc, y_val, "VALIDACIÓN")
acc_te, f1_te = evaluar_conjunto(X_test_sc, y_test, "TEST")

print("DETALLE FINAL (TEST):")
print(classification_report(y_test, modelo_lineal.predict(X_test_sc), target_names=le.classes_))

RESULTADOS TRAIN 
Accuracy: 0.9771 (97.71%)
F1-Score (Macro): 0.9697

RESULTADOS VALIDACIÓN 
Accuracy: 0.9867 (98.67%)
F1-Score (Macro): 0.9830

RESULTADOS TEST 
Accuracy: 0.9667 (96.67%)
F1-Score (Macro): 0.9325

DETALLE FINAL (TEST):
              precision    recall  f1-score   support

        High       0.89      0.81      0.85        21
         Low       0.99      0.99      0.99       153
      Medium       0.95      0.97      0.96       126

    accuracy                           0.97       300
   macro avg       0.94      0.92      0.93       300
weighted avg       0.97      0.97      0.97       300



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


### Conclusión final
En este notebook hemos comprobado que la Regresión Logística es un modelo que  se adapta bastante bien a este problema. Con una estructura muy sencilla, de solo 33 parámetros, hemos conseguido un 96,67% de acierto en los datos de prueba y un F1-Score de 0,9325.

La pequeña diferencia entre los resultados de entrenamiento (97,71%) y los de prueba (96,67%) muestra que el modelo no está memorizando los datos, sino que realmente ha aprendido el patrón general. Además, esto sugiere que las variables del conjunto de datos (fatiga, horas de sueño y horas de trabajo) tienen una relación bastante clara y directa con el riesgo de burnout.
